In [4]:
import numpy as np
import pandas as pd


# 1. Generate the random data (1000 rows, 4 columns)
Data = np.random.randint(1, 1000, (1000, 4))

# 2. Create the DataFrame with column names
df1 = pd.DataFrame(Data, columns=["Open", "High", "Low", "Close"])


# 3. Define the Signal Generation Function
def generate_ma_crossover_signals(
    data: pd.DataFrame, fast_window: int, slow_window: int
) -> pd.DataFrame:
    """Generates trading signals based on a Moving Average Crossover strategy.

    Parameters:
    - data: pd.DataFrame containing a 'Close' price column.
    - fast_window: The period for the short-term moving average (e.g., 50).
    - slow_window: The period for the long-term moving average (e.g., 200).

    Returns:
    - pd.DataFrame with added MA and Signal columns.
    """
    # FIX: Use the 'data' argument passed to the function, not the global 'df1'
    df = data.copy()

    # Calculate the Moving Averages
    df["Fast_MA"] = df["Close"].rolling(window=fast_window).mean()
    df["Slow_MA"] = df["Close"].rolling(window=slow_window).mean()

    # Create a position flag: 1 when Fast is above Slow, 0 otherwise
    df["Position"] = np.where(df["Fast_MA"] > df["Slow_MA"], 1, 0)

    # The signal is the day-to-day change in position
    # +1 indicates Fast crossed ABOVE Slow (BUY)
    # -1 indicates Fast crossed BELOW Slow (SELL)
    df["Signal"] = df["Position"].diff()

    # Clean up the NaN rows caused by the diff() and fill with 0 (Hold)
    df["Signal"] = df["Signal"].fillna(0).astype(int)

    # Clean up the intermediate 'Position' column
    df.drop(columns=["Position"], inplace=True)

    return df


# 4. Run the strategy (e.g., 50-day and 200-day MA)
df_signals = generate_ma_crossover_signals(df1, fast_window=50, slow_window=200)

# 5. View the rows where a trade was triggered
df_signals[df_signals["Signal"] != 0].head()

,Open,High,Low,Close,Fast_MA,Slow_MA,Signal
199,870,883,409,232,497.68,490.580,1
202,36,124,41,165,479.40,484.360,-1
204,470,705,599,779,492.38,484.485,1
207,950,61,428,192,478.74,488.655,-1
213,102,500,474,790,506.62,496.940,1
